In [9]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
SteganoGAN inference test script.

Loads a trained model and verifies encode → decode round-trips.
"""

import os
import sys
from typing import List

os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

from steganogan import SteganoGAN

In [10]:
# ── Config ────────────────────────────────────────────────────────────────────

MODEL_PATH        = "models/weights.steg"
INPUT_PATH        = "input.png"
OUTPUT_PATH       = "output.png"
MESSAGE           = "This is a super secret message!"
USE_GPU           = True
VERBOSE           = True
TEST_MULTIPLE     = True
CALLBACK_IMG_DIR  = "data/callback_images"

In [11]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def print_model_info(model: SteganoGAN) -> None:
    p = model.parameter_count
    print(f"\n{'=' * 60}")
    print("Model")
    print("=" * 60)
    print(f"  {model}")
    print(f"\n  Architecture:")
    print(f"    Encoder : {model.encoder}")
    print(f"    Decoder : {model.decoder}")
    if model.critic is not None:
        print(f"    Critic  : {model.critic}")
    print(f"\n  Parameters:")
    print(f"    Encoder : {p['encoder']:>12,}")
    print(f"    Decoder : {p['decoder']:>12,}")
    if model.critic is not None:
        print(f"    Critic  : {p['critic']:>12,}")
    print(f"    Total   : {p['total']:>12,}")


def test_roundtrip(
    model:        SteganoGAN,
    cover_path:   str,
    output_path:  str,
    message:      str,
) -> bool:
    """Encode *message* into *cover_path*, decode it, return success flag."""
    if not os.path.exists(cover_path):
        print(f"  ✗ Cover image not found: {cover_path!r}")
        return False

    model.encode(cover_path, output_path, message)
    decoded = model.decode(output_path)
    ok = decoded == message
    print(f"  {'✓' if ok else '✗'}  '{message[:60]}'"
          + ("" if ok else f"\n      got: '{decoded[:60]}'"))
    return ok


def test_batch(
    model:      SteganoGAN,
    image_dir:  str,
    messages:   List[str],
) -> None:
    """Run a round-trip test for each image in *image_dir* with a different message."""
    print(f"\n{'=' * 60}")
    print(f"Batch test  (dir: {image_dir!r})")
    print("=" * 60)

    if not os.path.exists(image_dir):
        print(f"  ✗ Image directory not found: {image_dir!r}")
        return

    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
    images = sorted([
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if os.path.splitext(f)[1].lower() in exts
    ])

    if not images:
        print(f"  ✗ No images found in {image_dir!r}")
        return

    print(f"  Found {len(images)} image(s), {len(messages)} message(s)")

    results = []
    for i, cover_path in enumerate(images):
        msg = messages[i % len(messages)]
        tmp = f"_test_tmp_{i}.png"
        fname = os.path.basename(cover_path)
        print(f"\n  [{i+1}/{len(images)}] {fname}")
        print(f"    msg : '{msg[:60]}'")
        try:
            ok = test_roundtrip(model, cover_path, tmp, msg)
            results.append(ok)
        finally:
            if os.path.exists(tmp):
                os.remove(tmp)

    passed = sum(results)
    total  = len(results)
    print(f"\n  Result: {passed}/{total} passed")

In [12]:
# ── Main ──────────────────────────────────────────────────────────────────────

print("=" * 60)
print("SteganoGAN Inference Test")
print("=" * 60)
print(f"  model  : {MODEL_PATH}")
print(f"  input  : {INPUT_PATH}")
print(f"  output : {OUTPUT_PATH}")

model = SteganoGAN.load(MODEL_PATH, gpu=USE_GPU, verbose=VERBOSE)
print_model_info(model)

print(f"\n{'=' * 60}")
print("Round-trip test")
print("=" * 60)
test_roundtrip(model, INPUT_PATH, OUTPUT_PATH, MESSAGE)

if TEST_MULTIPLE:
    test_batch(model, CALLBACK_IMG_DIR, messages=[
        "Hello World!",
        "SteganoGAN is working!",
        "1234567890",
        "Testing multiple messages",
        "Short msg",
        "Another secret payload",
        "Deep learning steganography",
        "PhD dissertation test",
    ])

print(f"\n{'=' * 60}")
print("Done.")
print("=" * 60)

SteganoGAN Inference Test
  model  : models/weights.steg
  input  : input.png
  output : output.png
Using Apple GPU (MPS).
Loading checkpoint: models/weights.steg → mps
Checkpoint loaded successfully.

Model
  SteganoGAN(encoder=DenseEncoder, decoder=DenseDecoder, critic=BasicCritic, data_depth=1, device=mps, params=72,224)

  Architecture:
    Encoder : DenseEncoder(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(33, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(65, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(97, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)
    Decoder : DenseDecoder(
  (conv1): Conv2d(3, 32, k